In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("aula_05_particoes")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 16:38:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv("/workspace/datasets/05_particoes/clientes.csv", header= True, inferSchema= True)

In [4]:
df.rdd.getNumPartitions()

1

In [5]:
df.explain(True)

== Parsed Logical Plan ==
Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspace/datasets/05_particoes/clientes.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<cliente_id:int,n

In [6]:
print(df.rdd.getNumPartitions())

1


In [7]:
df.show()

+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|cliente_id|      nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|
+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|         1| Cliente 1|   32|Feira de Santana|    BA|   Engenheiro|    12925.67|   M|false|   2025-02-22|
|         2| Cliente 2|   32|Feira de Santana|    BA|   Engenheiro|     4852.23|   M|false|   2024-01-16|
|         3| Cliente 3|   29|Feira de Santana|    BA|   Engenheiro|      8919.6|   F|false|   2023-02-08|
|         4| Cliente 4|   64|        Salvador|    BA|     Analista|     4253.53|   F|false|   2023-01-03|
|         5| Cliente 5|   60|        Salvador|    BA|   Engenheiro|     2710.92|   M|false|   2023-09-15|
|         6| Cliente 6|   30|        Salvador|    BA|   Engenheiro|    10282.08|   F|false|   2023-01-29|
|         7| Cliente 7|   46|Feira de Santana|

In [8]:
df_repartition = df.repartition(4)

In [9]:
df_repartition.rdd.getNumPartitions()

4

In [10]:
df_repartition.show()

+----------+-----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|cliente_id|       nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|
+----------+-----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|       225|Cliente 225|   21|       São Paulo|    SP|Desenvolvedor|    12893.39|   F|false|   2023-03-27|
|       201|Cliente 201|   67|        Salvador|    BA|Desenvolvedor|     1905.65|   F|false|   2023-10-17|
|       109|Cliente 109|   49|       São Paulo|    SP|    Professor|    12791.96|   M|false|   2023-07-30|
|       206|Cliente 206|   65|       São Paulo|    SP|Desenvolvedor|     14243.3|   F|false|   2023-10-24|
|        18| Cliente 18|   58|       São Paulo|    SP|    Professor|     5460.96|   M|false|   2025-05-08|
|        99| Cliente 99|   48|Feira de Santana|    BA|   Engenheiro|     7330.62|   F| true|   2025-02-20|
|       300|Cliente 300|   44|       

In [11]:
df_repartition.explain(True)

== Parsed Logical Plan ==
Repartition 4, true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Repartition 4, true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Repartition 4, true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ShuffleQueryStage 0
   +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=40]
      +- FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25

In [12]:
df_coalesce = df_repartition.coalesce(2)

In [13]:
df_coalesce.rdd.getNumPartitions()

2

In [14]:
df_coalesce.explain(True)

== Parsed Logical Plan ==
Repartition 2, false
+- Repartition 4, true
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Repartition 2, false
+- Repartition 4, true
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Repartition 2, false
+- Repartition 4, true
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   Coalesce 2
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=85]
         +- FileSc

In [ ]:
spark.stop()